In [1]:
## WORKSPACE :
# -----------
# Input data files are available in the read-only "../input/" directory

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/tmp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session


from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

INPUT = Path("/kaggle/input/")
DATA_ROOT = INPUT / "competitions/rsna-knee-abnormality-detection"

"""
DATA HIERARCHY: 
***************

[*] TRAINING SET:
    -------------
    - train.csv : 
        The outermost layer, where each row is a complete study with a unique StudyInstanceUID and a multilingual narrative Report. 
        There are 12 label columns which will be used as target columns of the image classifier, 
        but only a fraction of studies are pre-labelled.
        
    - train_series.csv : 
        Each study consists of multiple MRI series;
        e.g. side-to-side (sagittal), top-to-bottom (axial), front-to-back (coronal) with different fluid and fat sensitivity settings.
        Here, each row is a unique (StudyInstanceUID, SeriesInstanceUID) with its corresponding flags for MRI settings.
        
    - train_series : WARNING : 500GB+
        This is where the actual MRI images are stored.
        - One folder for each StudyInstanceUID in train.csv.
            - One subfolder for each (StudyInstanceUID, SeriesInstanceUID) in train_series.csv.
              Each such subfolder consists of multiple DICOM files;
                - one file represents an MRI slice.
        
[*] TEST SET: For the test set, we do not have any written narrative. The image-data hierarchy remains the same.
    ---------

OBJECTIVE: Predict the probability of 12 abnormality classes given MRI imagery.
**********
[Step 1] Populate the TARGETS based on the radiology Report.
[Step 2] Train a multi-label classifier to predict the TARGETS from the corresponding DICOM images.


[[THE CEILING OF STEP-1 DETERMINES THE FLOOR OF STEP-2]]
********************************************************
[*] The annotated reference set (58 StudyInstanceUIDs) is <2% of the total training set (4407 StudyInstanceUIDs).
The LLM extractor works solely on the Report text, 
while the reference set is annotated based on the original MRI images by an expert independent of the radiologist who prepared the Report.
Some information available in the images was lost during the process of transcription by the radiologist,
and hence the ground truth of MRI images can never be 100% recovered from reports alone.
e.g., Of the 27 cases positive for Synovitis in the reference set, only ~1/3 of the corresponding reports mention it explicitly, while Synovitis shows a strong co-occurrence with Effusion. 
Upon further investigation, (see https://www.sciencedirect.com/science/article/pii/S1063458410002438, https://www.sciencedirect.com/science/article/pii/S266591312500038X)
this may be explained by the fact that, on non-contrast MRI, 
radiologists may not always be able to cleanly distinguish distended joint fluid from thickened synovium; 
both can appear hyperintense, potentially introducing human-judgement bias.

[*] The final image-classifier model (step-2) propagates classification errors from the LLM-generated pseudolabels (step-1), 
while eventually introducing additional prediction errors of its own.

"""

TARGETS = pd.read_csv(
    DATA_ROOT / "sample_submission.csv",
    nrows=0, index_col="StudyInstanceUID",
).columns.tolist()

train = pd.read_csv(
    DATA_ROOT / "train.csv",
    index_col="StudyInstanceUID",
    dtype={"Report":'string'}|{col:np.float32 for col in TARGETS},
    # we will use soft labels instead of hard 0-1 labeling
)
train = train.set_index(
    pd.MultiIndex.from_arrays(
        [train[TARGETS].notna().any(axis=1).values, train.index],
        names=["label_mask", "StudyInstanceUID"],
    )
).sort_index()
LABELLED = train.loc[(True,), :]
assert LABELLED[TARGETS].notna().to_numpy().all(), "a labelled study has a missing class; astype(bool) would turn NaN into True"
LABELLED = LABELLED.astype({col:bool for col in TARGETS})
n_LABELLED = len(LABELLED)


train_series = pd.read_csv(
    DATA_ROOT / "train_series.csv",
    index_col=["StudyInstanceUID", "SeriesInstanceUID"],
    dtype={
        "StudyInstanceUID":'string', "SeriesInstanceUID":'string',
        "Anatomical_Plane":'category', "Fluid_Sensitive":bool, "Fat_Suppression":bool,
    }, 
    # only 2-3 unique labels for each column of len ~25k
).sort_index()


print("\ntrain: ", train.shape)
display(train)
print("\ntrain_series: ", train_series.shape)
display(train_series)
print("\ntargets: ", TARGETS)
print("\nStudies with labels: ", n_LABELLED)
display(
    (100 * LABELLED[TARGETS].mean(axis=0)).round(2).to_frame("label_positive (%)").T
    # label_negative (%) = 100 - label_positive (%), since labelled studies have no NaN.
)
print("\nStudies without labels: ", len(train) - n_LABELLED)
print("\nReport char-counts: ", train["Report"].str.len().describe().astype(np.int32).to_dict())


# NOTE: This competition only rewards macro-AUC.
# *****
# [PROS] as long as the model preserves the ordinal relationship between POSITIVE/NEGATIVE, that suffices;
# prediction probability calibration is not required here.
# [CONS] it treats each of the 12 classes equally,
# rather than allowing high-volume classes to dominate the final score.



train:  (4407, 13)


Report  \
label_mask StudyInstanceUID                                                                                        
False      1.2.826.0.1.3680043.8.498.100048732290990538690...  Técnica: RMN de la rodilla. Resultados: Rotura...   
           1.2.826.0.1.3680043.8.498.100049459274726560271...  [DATE]: * MR Knie Rechts 15ch AA Klinische Inl...   
           1.2.826.0.1.3680043.8.498.100092786926066315735...  Hallazgos:\nNo hay alteraciones en significati...   
           1.2.826.0.1.3680043.8.498.100096392031707502741...   In the medial compartment, the meniscus is no...   
           1.2.826.0.1.3680043.8.498.100136637424007360294...  CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...   
...                                                                                                          ...   
True       1.2.826.0.1.3680043.8.498.869686002397243106789...  Right knee pain. R/O ACL insufficiency. \nMRI ...   
           1.2.826.0.1.3680043.8.498.880774186393011744099...  Uznapredovale goanrtrotske promjene uz opsežan...   
           1.2.826.0.1.3680043.8.498.902835653810420817685...  Técnica: RMN de la rodilla. Resultados: Cambio...   
           1.2.826.0.1.3680043.8.498.944337534718903061080...  No significant knee effusion. Normal medial an...   
           1.2.826.0.1.3680043.8.498.972747202576345840715...  Stigmata der Inaktivitätsosteopenie sowie verb...   

                                                               ACL  MCL  \
label_mask StudyInstanceUID                                               
False      1.2.826.0.1.3680043.8.498.100048732290990538690...  NaN  NaN   
           1.2.826.0.1.3680043.8.498.100049459274726560271...  NaN  NaN   
           1.2.826.0.1.3680043.8.498.100092786926066315735...  NaN  NaN   
           1.2.826.0.1.3680043.8.498.100096392031707502741...  NaN  NaN   
           1.2.826.0.1.3680043.8.498.100136637424007360294...  NaN  NaN   
...                                                            ...  ...   
True       1.2.826.0.1.3680043.8.498.869686002397243106789...  1.0  0.0   
           1.2.826.0.1.3680043.8.498.880774186393011744099...  1.0  0.0   
           1.2.826.0.1.3680043.8.498.902835653810420817685...  0.0  0.0   
           1.2.826.0.1.3680043.8.498.944337534718903061080...  0.0  0.0   
           1.2.826.0.1.3680043.8.498.972747202576345840715...  1.0  0.0   

                                                               Medial Meniscus  \
label_mask StudyInstanceUID                                                      
False      1.2.826.0.1.3680043.8.498.100048732290990538690...              NaN   
           1.2.826.0.1.3680043.8.498.100049459274726560271...              NaN   
           1.2.826.0.1.3680043.8.498.100092786926066315735...              NaN   
           1.2.826.0.1.3680043.8.498.100096392031707502741...              NaN   
           1.2.826.0.1.3680043.8.498.100136637424007360294...              NaN   
...                                                                        ...   
True       1.2.826.0.1.3680043.8.498.869686002397243106789...              1.0   
           1.2.826.0.1.3680043.8.498.880774186393011744099...              1.0   
           1.2.826.0.1.3680043.8.498.902835653810420817685...              0.0   
           1.2.826.0.1.3680043.8.498.944337534718903061080...              0.0   
           1.2.826.0.1.3680043.8.498.972747202576345840715...              1.0   

                                                               Lateral Meniscus  \
label_mask StudyInstanceUID                                                       
False      1.2.826.0.1.3680043.8.498.100048732290990538690...               NaN   
           1.2.826.0.1.3680043.8.498.100049459274726560271...               NaN   
           1.2.826.0.1.3680043.8.498.100092786926066315735...               NaN   
           1.2.826.0.1.3680043.8.498.100096392031707502741...               NaN   
           1.2.826.0.1.3680043.8.498.1001366374240073602


train_series:  (24371, 3)


Fluid_Sensitive  \
StudyInstanceUID                                   SeriesInstanceUID                                                     
1.2.826.0.1.3680043.8.498.100048732290990538690... 1.2.826.0.1.3680043.8.498.123431101950362134834...             True   
                                                   1.2.826.0.1.3680043.8.498.138212297449972206415...             True   
                                                   1.2.826.0.1.3680043.8.498.230848365367225952758...            False   
                                                   1.2.826.0.1.3680043.8.498.407342061024587230961...             True   
                                                   1.2.826.0.1.3680043.8.498.757148999972036157840...            False   
...                                                                                                                ...   
1.2.826.0.1.3680043.8.498.999576112073130986961... 1.2.826.0.1.3680043.8.498.543090017839016684838...             True   
                                                   1.2.826.0.1.3680043.8.498.628702981296394013441...             True   
                                                   1.2.826.0.1.3680043.8.498.750798929395794071069...            False   
                                                   1.2.826.0.1.3680043.8.498.753175095197121489120...             True   
                                                   1.2.826.0.1.3680043.8.498.957222785063165420178...             True   

                                                                                                       Fat_Suppression  \
StudyInstanceUID                                   SeriesInstanceUID                                                     
1.2.826.0.1.3680043.8.498.100048732290990538690... 1.2.826.0.1.3680043.8.498.123431101950362134834...             True   
                                                   1.2.826.0.1.3680043.8.498.138212297449972206415...             True   
                                                   1.2.826.0.1.3680043.8.498.230848365367225952758...            False   
                                                   1.2.826.0.1.3680043.8.498.407342061024587230961...             True   
                                                   1.2.826.0.1.3680043.8.498.757148999972036157840...            False   
...                                                                                                                ...   
1.2.826.0.1.3680043.8.498.999576112073130986961... 1.2.826.0.1.3680043.8.498.543090017839016684838...             True   
                                                   1.2.826.0.1.3680043.8.498.628702981296394013441...             True   
                                                   1.2.826.0.1.3680043.8.498.750798929395794071069...            False   
                                                   1.2.826.0.1.3680043.8.498.753175095197121489120...             True   
                                                   1.2.826.0.1.3680043.8.498.957222785063165420178...             True   

                                                                                                      Anatomical_Plane  
StudyInstanceUID                                   SeriesInstanceUID                                                    
1.2.826.0.1.3680043.8.498.100048732290990538690... 1.2.826.0.1.3680043.8.498.123431101950362134834...         Sagittal  
                                                   1.2.826.0.1.3680043.8.498.138212297449972206415...            Axial  
                                                   1.2.826.0.1.3680043.8.498.230848365367225952758...          Coronal  
                                                   1.2.826.0.1.3680043.8.498.407342061024587230961...          Coronal  
                                                   1.2.826.0.1.3680043.8.498.757148999972036157840...         Sagittal  
...                                                                               


targets:  ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']

Studies with labels:  58


,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
label_positive (%),41.38,15.52,44.83,39.66,25.86,18.97,36.21,60.34,46.55,20.69,32.76,31.03



Studies without labels:  4349

Report char-counts:  {'count': 4407, 'mean': 1097, 'std': 693, 'min': 52, '25%': 587, '50%': 977, '75%': 1459, 'max': 4743}
